In [ ]:
from typing import List

import numpy as np
import pandas as pd
import scipy.stats
import sklearn.metrics
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split

from sportsml.cbb.data.features import STATS_COLUMNS
from sportsml.utils.stats import process_averages


In [ ]:
df = pd.read_csv("../data/cbb/raw.csv")

In [ ]:
GAME_META_COLS = [
    "Season",
    # "DayNum",
    # "TeamID",
    # "TeamID_OPP",
    # "Loc",
    # "Score",
    # "Score_OPP",
    "PlusMinus",
]

In [ ]:
avgs = process_averages(
    df,
    stats_columns=STATS_COLUMNS,
    game_meta_columns=GAME_META_COLS,
    season_column="Season",
    date_column="DayNum",
    team_column="TeamID",
    team_opp_column="TeamID_OPP",
    rolling_windows=[3],
    use_all_data=False,
    avg_suffix="_AVG",
    rolling_suffix="_ROLLING",
    opp_prefix="OPP_",
).dropna()

In [ ]:
train = avgs[avgs["Season"] < 2025]
test = avgs[avgs["Season"] == 2025]

In [ ]:
y_train = train.pop("PlusMinus").values
X_train = train.drop(columns=["Season"]).values
y_test = test.pop("PlusMinus").values
X_test = test.drop(columns=["Season"]).values

In [ ]:
y = avgs.pop("PlusMinus").values
X = avgs.values

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
rf = RandomForestRegressor(n_estimators=25, random_state=42, verbose=4)
rf.fit(X_train, y_train)

In [ ]:
preds = rf.predict(X_test)
stats = {
    "rmse": sklearn.metrics.root_mean_squared_error(y_test, preds),
    "r2": sklearn.metrics.r2_score(y_test, preds),
    "mae": sklearn.metrics.mean_absolute_error(y_test, preds),
    "accuracy": sklearn.metrics.accuracy_score(y_test > 0, preds > 0),
    "precision": sklearn.metrics.precision_score(y_test > 0, preds > 0),
    "recall": sklearn.metrics.recall_score(y_test > 0, preds > 0),
    "f1": sklearn.metrics.f1_score(y_test > 0, preds > 0),
    "spearmanr": scipy.stats.spearmanr(y_test, preds)[0],
    "pearsonr": scipy.stats.pearsonr(y_test, preds)[0],
}

In [ ]:
pd.Series(stats)

In [ ]:
pd.DataFrame({'preds': preds, 'true': y_test}).plot.scatter(x="true", y="preds")